1. LangChain
    
        Prompts
        LLMs
        Chains
        RAG
        Memory
        Tools
        Output Parsers

2. LangGraph

        Agents
        Multi-step workflows
        State management
        Checkpointing
        Human-in-the-loop

3. LangSmith

        Tracing
        Debugging
        Monitoring
        Evaluation

4. LangServe

        Deploy

In [69]:
import os
from dotenv import load_dotenv
load_dotenv()

#LANGSMITH

langapi=os.getenv("LANGCHAIN_API_KEY")
langproject=os.getenv("LANGCHAIN_PROJECT")
langtrace=os.getenv("LANGCHAIN_TRACING_V2")
hftoken=os.getenv("HF_TOKEN")

In [70]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import WebBaseLoader

webl=WebBaseLoader("https://huggingface.co/")
webload=webl.load()

In [71]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

ts=RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
textsplit=ts.split_documents(webload)
textsplit

[Document(metadata={'source': 'https://huggingface.co/', 'title': 'Hugging Face – The AI community building the future.', 'description': 'We’re on a journey to advance and democratize artificial intelligence through open source and open science.', 'language': 'No language found.'}, page_content='Hugging Face – The AI community building the future.'),
 Document(metadata={'source': 'https://huggingface.co/', 'title': 'Hugging Face – The AI community building the future.', 'description': 'We’re on a journey to advance and democratize artificial intelligence through open source and open science.', 'language': 'No language found.'}, page_content='Hugging Face        Models  Datasets  Spaces  Buckets new Docs  Enterprise  Pricing     Website'),
 Document(metadata={'source': 'https://huggingface.co/', 'title': 'Hugging Face – The AI community building the future.', 'description': 'We’re on a journey to advance and democratize artificial intelligence through open source and open science.', 'la

In [72]:
from langchain_huggingface import HuggingFaceEmbeddings

hf=HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
texts = [doc.page_content for doc in textsplit]
embeddoc=hf.embed_documents(texts)
embeddoc

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[[-0.05867841839790344,
  0.018056511878967285,
  0.0594027116894722,
  0.022063784301280975,
  0.0651833638548851,
  -0.0031388478819280863,
  -0.031139597296714783,
  -0.08126243203878403,
  -0.005076626315712929,
  -0.04316253960132599,
  0.009494051337242126,
  -0.06805124878883362,
  -0.0166391059756279,
  0.036444395780563354,
  0.07608997821807861,
  0.012086998671293259,
  -0.0024336541537195444,
  -0.019237762317061424,
  -0.02537519671022892,
  0.06330165266990662,
  -0.045540355145931244,
  0.04118315503001213,
  0.03168315067887306,
  -0.04293736442923546,
  -0.02691596746444702,
  0.040828119963407516,
  0.034766778349876404,
  -0.06385823339223862,
  0.0590970553457737,
  0.002359329955652356,
  0.03187388554215431,
  -0.029977913945913315,
  0.034658703953027725,
  0.06225036084651947,
  -0.03676515817642212,
  0.11842599511146545,
  -0.009385316632688046,
  0.06803324073553085,
  -0.027545219287276268,
  -0.05308305472135544,
  -0.07320033013820648,
  -0.035231567919254

In [73]:
from langchain_chroma import Chroma

db=Chroma.from_documents(textsplit,hf)

In [74]:
retriever=db.as_retriever()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt=ChatPromptTemplate.from_messages(
    [
        ("system",
         """act like a chatgpt & claude pro model answer all the things correctly and at your best without any mistake 
         context {t}"""),
        MessagesPlaceholder(variable_name="history"),
        ("human","{input}")
    ]
)

In [76]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

In [77]:
from langchain_ollama import ChatOllama
from langchain_core.messages import trim_messages

model=ChatOllama(model="llama3:8b")
chain=prompt|model
chain=RunnableWithMessageHistory(chain,get_session_history, input_messages_key="input", history_messages_key="history")
trim=trim_messages(strategy="last", max_tokens=200, token_counter=model, include_system=True, )

username=input("enter user name: ")
session_id=username

while True:
    userinput=input("enter questions: ")
    print("enter stop to stop conversations")
    if userinput.lower()=="stop":
        break
    
    history=get_session_history(session_id)
    history.messages=trim.invoke(history.messages)

    t=retriever.invoke(userinput)
    response=chain.invoke({"input":userinput,
                           "t": t},
                 config={"configurable":{"session_id":session_id}})
    print(response)

history=get_session_history(session_id)
print(len(history.messages))

trimmed_messages = trim.invoke(history.messages)
print(len(trimmed_messages))

c:\Users\Nalini\OneDrive\Desktop\langchain\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


enter stop to stop conversations
content="I'm ChatGPT, an AI chatbot trained by a large language model, and Claude Pro is another AI model that I've been trained alongside! We're both here to help answer your questions to the best of our abilities.\n\nWhat would you like to talk about? Do you have any specific questions or topics in mind that you'd like us to discuss?" additional_kwargs={} response_metadata={'model': 'llama3:8b', 'created_at': '2026-06-21T09:32:57.5655452Z', 'done': True, 'done_reason': 'stop', 'total_duration': 123184165900, 'load_duration': 29415970300, 'prompt_eval_count': 442, 'prompt_eval_duration': 64043162000, 'eval_count': 74, 'eval_duration': 29624529000, 'logprobs': None, 'model_name': 'llama3:8b', 'model_provider': 'ollama'} id='lc_run--019ee984-c9e5-7d40-9c30-d4f7aa767ddb-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 442, 'output_tokens': 74, 'total_tokens': 516}
enter stop to stop conversations


KeyboardInterrupt: 

### Streamlit

In [ ]:
import streamlit as st

st.set_page_config(
    page_title="AI Assistant",
    page_icon="🤖",
    layout="wide"
)

st.title("🤖 AI Assistant")

with st.sidebar:

    st.header("Knowledge Base")

    uploaded_files = st.file_uploader(
        "Upload Documents",
        accept_multiple_files=True,
        type=["pdf","docx","txt","csv"]
    )

    website_url = st.text_input(
        "Website URL"
    )

    if st.button("Process Documents"):
        pass

    st.divider()

    if st.button("Clear Chat"):
        st.session_state.messages=[]

chat_container = st.container()

prompt = st.chat_input("Ask Anything...")